# Scam Triage AI 🚨
**Track: Finance** — Veritas AI International High School Hackathon

**What it does:** Paste in any text message, email, or DM. In under a second, Scam Triage AI gives it a **Scam Risk Score (0–100)**, lists the *specific red flags* it found (not just "this looks fake"), and tells you exactly what to do next. It's built to catch the scams that are actually hitting people our age right now: fake package delivery texts, fake bank fraud alerts, fake job offers, and social-media investment/romance scams.

**Why it matters:** Americans lost a record $15.9 billion to scams in 2025, and text messages were the single most common way scammers made first contact that year (FTC). Consumers specifically lost $470 million to text scams in 2024 — five times what was lost in 2020 — and young adults (18–29) saw the sharpest rise in scam-text attempts of any age group, with reports roughly tripling (Consumer Reports, 2025). People our age aren't scammed because we're careless — we just haven't been shown exactly what the red flags look like. This tool shows the *why*, every time.

**How it works:** We combine two things, run entirely for free with no API key:
1. A **zero-shot NLP model** (`facebook/bart-large-mnli`, via Hugging Face `transformers`) that reads the message and scores how well it matches the idea "this is a scam / phishing attempt" vs. "this is a normal, legitimate message" — without ever being trained on labeled scam data.
2. A transparent **red-flag rule engine** we wrote ourselves, which scans for the exact patterns real scams use: urgency language, payment-method requests (gift cards, crypto, wire transfers), impersonation of a bank/government agency, suspicious/shortened links, and more.

Combining both means the tool is never a black box — it always shows *which specific signals* fired, which is exactly what makes someone trust (and learn from) the result.

**Run every cell below, top to bottom.**

## 1. Setup

In [ ]:
!pip install -q transformers torch gradio

## 2. Load the zero-shot model
`facebook/bart-large-mnli` is a free, open model made for exactly this: scoring text against any labels you give it, with no training required. First load takes ~1–2 minutes on Colab.

In [ ]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Model loaded.")

## 3. The red-flag rule engine
Each rule below is a real, documented pattern from FTC scam reporting (fake delivery alerts, fake fraud alerts, job scams, gift-card/crypto payment requests, urgency tactics). Each one that fires adds points to the score and gets named in the output — this is what makes the result explainable instead of a mystery number.

**This is the part you should personally tweak** — add your own rule, change a weight, add an example — so you can genuinely explain what you added yourself.

In [ ]:
import re

RULES = [
    ("Urgency / scare tactics", r"\b(act now|immediately|urgent|right away|within (24|48) hours|account will be (closed|suspended|locked)|final notice|act fast)\b", 15),
    ("Asks for gift cards, crypto, or wire transfer", r"\b(gift card|bitcoin|crypto|wire transfer|western union|cash app|zelle|venmo)\b.*\b(pay|send|purchase|buy)\b|\b(pay|send|purchase|buy)\b.*\b(gift card|bitcoin|crypto|wire transfer)\b", 30),
    ("Fake package / delivery alert", r"\b(package|delivery|parcel|shipment)\b.*\b(fee|customs|redeliver|confirm|unable to deliver|held)\b", 20),
    ("Impersonates a bank, government agency, or employer", r"\b(irs|social security|ssa|your bank|chase|wells fargo|paypal security|amazon security|hr department)\b", 15),
    ("Suspicious or shortened link", r"https?://(bit\.ly|tinyurl|t\.co|is\.gd|[a-z0-9]{6,}\.(xyz|top|click|info))\S*|\b\w+-\w+\.(net|com)/[a-z0-9]{6,}", 20),
    ("Unrealistic job offer or prize", r"\b(you('ve| have) (won|been selected)|guaranteed income|work from home.*\$\d{3,}|no experience.*\$\d{2,} ?/ ?(hr|hour|day))\b", 20),
    ("Asks for personal/financial info directly", r"\b(verify your (account|identity|ssn|social security)|enter your (password|pin|card number))\b", 25),
    ("Generic greeting (no real name used)", r"^(dear (customer|user|valued customer|sir/madam))\b", 8),
    ("'Wrong number' opener (common scam-chat starter)", r"\b(hi|hey),? (is this|this is) .*\?\s*(sorry )?wrong number\b", 15),
]

def run_rule_engine(text: str):
    text_lower = text.lower()
    hits = []
    points = 0
    for name, pattern, weight in RULES:
        if re.search(pattern, text_lower):
            hits.append(name)
            points += weight
    return hits, min(points, 70)  # cap rule-engine contribution at 70

## 4. Combine both signals into one Scam Risk Score

In [ ]:
SCAM_LABELS = [
    "a scam, phishing attempt, or fraudulent message trying to trick someone",
    "a normal, legitimate message from a real person or real business",
]

def score_message(text: str):
    """Pure scoring logic, no UI formatting. Returns (score, model_prob, hits list)."""
    zs_result = zero_shot(text, candidate_labels=SCAM_LABELS)
    scam_label_idx = zs_result["labels"].index(SCAM_LABELS[0])
    model_scam_prob = zs_result["scores"][scam_label_idx]  # 0-1

    hits, rule_points = run_rule_engine(text)

    final_score = round(model_scam_prob * 30 + rule_points)
    final_score = min(final_score, 100)
    return final_score, model_scam_prob, hits

## 5. Quick sanity check
Testing on a realistic fake-package scam text and a normal message, so you can see the scoring work before opening the live demo.

In [ ]:
test_scam = "Dear Customer, your package delivery failed due to an unpaid customs fee. Pay $2.99 immediately at http://usps-redelivery.xyz/pay or your package will be returned."
test_legit = "Hey, are we still meeting at 4pm for the group project today? Let me know if that time still works for you."

for label, msg in [("SCAM example", test_scam), ("LEGIT example", test_legit)]:
    score, model_prob, hits = score_message(msg)
    print(f"--- {label} ---")
    print(f"Score: {score}/100 | Model's own read: {model_prob*100:.0f}% scam-like")
    print("Flags:", hits if hits else "none")
    print()

## 6. Live demo (polished UI)
A custom `gr.Blocks` layout with a colored risk gauge, a clear verdict, and the exact red flags listed — built to look good on a demo recording, not just function. Click a built-in example to see it work instantly, or paste your own message.

In [ ]:
import gradio as gr

def risk_band(score):
    if score >= 60:
        return "HIGH RISK", "#dc2626", "#fee2e2", "Don't click any links or reply. Delete it, and if it claims to be your bank, employer, or a delivery company, contact them directly using a number or site you already know is real."
    elif score >= 30:
        return "MEDIUM RISK", "#d97706", "#fef3c7", "Be cautious. Verify independently before clicking links, replying, or sending any information or money."
    else:
        return "LOW RISK", "#16a34a", "#dcfce7", "Looks relatively safe — but if something still feels off, trust your gut and verify independently anyway."

def analyze_message(text: str):
    if not text or not text.strip():
        empty = "<div style='padding:20px;color:#6b7280;font-family:sans-serif;'>Paste a message above (or click an example below) to see the result here.</div>"
        return empty, ""

    score, model_prob, hits = score_message(text)
    label, color, bg, advice = risk_band(score)

    gauge_html = f"""
    <div style='font-family:-apple-system,Segoe UI,sans-serif;padding:20px;border-radius:16px;background:{bg};border:2px solid {color};'>
      <div style='display:flex;justify-content:space-between;align-items:baseline;'>
        <span style='font-size:15px;font-weight:600;color:{color};letter-spacing:0.5px;'>{label}</span>
        <span style='font-size:34px;font-weight:800;color:{color};'>{score}<span style='font-size:16px;color:#6b7280;'>/100</span></span>
      </div>
      <div style='height:10px;background:#e5e7eb;border-radius:6px;margin-top:10px;overflow:hidden;'>
        <div style='height:100%;width:{score}%;background:{color};border-radius:6px;transition:width 0.4s;'></div>
      </div>
      <div style='margin-top:14px;font-size:14px;color:#374151;line-height:1.5;'>
        💡 <b>What to do:</b> {advice}
      </div>
      <div style='margin-top:10px;font-size:12px;color:#9ca3af;'>AI model's own read: {model_prob*100:.0f}% likely a scam, from message content alone.</div>
    </div>
    """

    flags_html = "".join(
        f"<li style='margin-bottom:6px;'>{h}</li>" for h in hits
    ) if hits else "<li style='color:#9ca3af;'>No specific red-flag patterns matched.</li>"
    flags_full_html = f"""
    <div style='font-family:-apple-system,Segoe UI,sans-serif;padding:16px 20px;'>
      <div style='font-size:13px;font-weight:600;color:#374151;margin-bottom:8px;'>RED FLAGS DETECTED</div>
      <ul style='margin:0;padding-left:20px;color:#1f2937;font-size:14px;'>{flags_html}</ul>
    </div>
    """
    return gauge_html, flags_full_html

CUSTOM_CSS = """
#title_md h1 { font-size: 30px !important; margin-bottom: 2px !important; }
#subtitle { color: #6b7280; font-size: 15px; margin-top: 0; margin-bottom: 18px; }
.gradio-container { max-width: 720px !important; margin: auto !important; }
footer { display: none !important; }
"""

# Layout order matters here: input -> button -> examples -> result, all stacked
# close together with no side-by-side columns, so nothing requires scrolling
# back and forth to use. Examples are wired with fn/outputs so clicking one
# runs the analysis immediately and the result appears right below it.
with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="red")) as demo:
    gr.Markdown("# 🚨 Scam Triage AI", elem_id="title_md")
    gr.Markdown(
        "Paste a message and click Analyze, or click an example below — "
        "either way, the result shows up right under this box.",
        elem_id="subtitle",
    )

    input_box = gr.Textbox(
        lines=5,
        label="Message to check",
        placeholder="Paste a suspicious text, email, or DM here...",
    )
    analyze_btn = gr.Button("🔍 Analyze", variant="primary", size="lg")

    gr.Markdown("**Result:**")
    gauge_out = gr.HTML()
    flags_out = gr.HTML()

    gr.Markdown("**Or try an example (runs automatically):**")
    gr.Examples(
        examples=[
            "Dear Customer, your package delivery failed due to an unpaid customs fee. Pay $2.99 immediately at http://usps-redelivery.xyz/pay or your package will be returned.",
            "Hi, is this Jordan? Sorry, wrong number! ...actually, how's your day going? :)",
            "Your Chase account has been temporarily suspended. Verify your identity immediately by entering your account PIN at the link below to restore access.",
            "Congratulations, you've been selected! Work from home, no experience needed, $85/hr guaranteed income. Reply now to claim your spot.",
            "Hey, are we still meeting at 4pm for the group project today?",
        ],
        inputs=input_box,
        outputs=[gauge_out, flags_out],
        fn=analyze_message,
        run_on_click=True,
        label="",
    )

    analyze_btn.click(fn=analyze_message, inputs=input_box, outputs=[gauge_out, flags_out])
    input_box.submit(fn=analyze_message, inputs=input_box, outputs=[gauge_out, flags_out])

demo.launch(share=True, debug=False)

## Credits & tools used
- **Model:** `facebook/bart-large-mnli`, via Hugging Face `transformers` — https://huggingface.co/facebook/bart-large-mnli
- **Demo UI:** Gradio — https://www.gradio.app/
- **Rule engine:** written entirely by our team, based on red-flag patterns documented by the FTC's text-scam spotlight reports.
- **AI coding assistance:** Claude (Anthropic) was used to help write, debug, and structure this notebook and the accompanying write-up.
- **Stats cited in write-up:** FTC 2025 fraud report ($15.9B total losses, text messages as top contact method); FTC text-scam spotlight ($470M lost to text scams in 2024, 5x 2020); Consumer Reports 2025 Cyber Readiness survey (scam-text attempts on 18–29-year-olds roughly tripling).
- No paid APIs, no labeled training dataset, no account sign-ups.

## What we'd add with more time
- Fine-tune a small classifier on a public phishing/SMS-spam dataset (e.g. the SMS Spam Collection dataset) to sharpen accuracy beyond zero-shot.
- Let users forward a screenshot (OCR'd) instead of pasting text.
- A browser extension / Messages app shortcut so the check happens where the text actually arrives.